In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
USE_KAGGLEHUB = True

In [ ]:
import os, json
dnepozitek_polyvore_outfits_path = kagglehub.dataset_download('dnepozitek/polyvore-outfits')
dataset_root = os.path.join(dnepozitek_polyvore_outfits_path, "polyvore_outfits")

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
mynrawu_met_activity_path = kagglehub.dataset_download('mynrawu/met-activity')
google_gemma_3_transformers_gemma_3_4b_it_1_path = kagglehub.model_download('google/gemma-3/Transformers/gemma-3-4b-it/1')

print('Data source import complete.')


In [ ]:
# === 1. 掛載 Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers
!pip install datasets
!pip install tqdm

In [ ]:
import os, json, base64, requests, math, glob, time, re
from typing import List, Dict, Any, Optional, Tuple
import unicodedata
import pandas as pd
from PIL import Image
from collections import defaultdict
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
from transformers import pipeline
from difflib import get_close_matches

# ========= 路徑與設定（改成 valid.json，移除 FITB） =========
MAKE_COLLAGE = True

OUT_JSON = "/content/drive/MyDrive/valid_met.json" # Changed from train_met.json

FEEDBACK_ON_FAIL = True
ALLOW_FUZZY = True
FUZZY_CUTOFF = 0.6

# CUDA 設定
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ===== MET 活動資料 =====
with open(os.path.join(mynrawu_met_activity_path, "adult_activity_compendium_sorted_2024.json"), "r", encoding="utf-8") as f:
    activity_data = json.load(f)

activity2met = {
    entry["ActivityDescription"]: entry["METValue"]
    for entry in activity_data
    if "ActivityDescription" in entry
}
activities_text = "\n".join(activity2met.keys())

# ===== Gemma-3 本地推論（多模態） =====
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
pipe = pipeline(
    task="image-text-to-text",
    model=google_gemma_3_transformers_gemma_3_4b_it_1_path,
    device="cuda",
    dtype=torch.bfloat16,
    trust_remote_code=True
)

# ===== Polyvore metadata 與影像路徑 =====
print("正在載入 Metadata / 設定資料集路徑...")
dataset_root = dnepozitek_polyvore_outfits_path
meta_path = os.path.join(dataset_root, "polyvore_outfits", "polyvore_item_metadata.json")
meta_df = pd.read_json(meta_path, orient="index")
meta_df.index = meta_df.index.astype(str)
meta_df.index.name = "item_id"

IMAGES_DIR = os.path.join(dataset_root, "polyvore_outfits", "images")
SPLIT_DIR   = os.path.join(dataset_root, "polyvore_outfits", "disjoint")

# ========== 工具函式 ==========
def image_path(item_id: str) -> Optional[str]:
    if item_id in meta_df.index:
        row = meta_df.loc[item_id]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        for key in ("image", "image_path", "filename", "img", "img_path", "img_filename"):
            v = row.get(key, None)
            if isinstance(v, str) and v.strip():
                p = os.path.join(IMAGES_DIR, v.lstrip("/")) if not os.path.isabs(v) else v
                if os.path.exists(p):
                    return p
    for ext in ("jpg","jpeg","png","JPG","JPEG","PNG"):
        p = os.path.join(IMAGES_DIR, f"{item_id}.{ext}")
        if os.path.exists(p): return p
    hits = glob.glob(os.path.join(IMAGES_DIR, "**", f"{item_id}.*"), recursive=True)
    return hits[0] if hits else None

def make_collage(image_paths: List[str], out_path: str, tile=(224,224), cols=4):
    if not image_paths: return None
    w,h = tile; rows = math.ceil(len(image_paths)/cols)
    canvas = Image.new("RGB", (cols*w, rows*h), (255,255,255))
    r=c=0
    for p in image_paths:
        try:
            im = Image.open(p).convert("RGB").resize(tile, Image.LANCZOS)
            canvas.paste(im, (c*w, r*h))
            c+=1
            if c==cols: c=0; r+=1
        except Exception as e:
            print("[WARN] 讀圖失敗：", p, e)
    canvas.save(out_path); return out_path

def _infer_from_text(row: pd.Series) -> str:
    tokens = []
    cats = row.get("categories", None)
    if isinstance(cats, list): tokens += [str(x).lower() for x in cats]
    elif isinstance(cats, str): tokens.append(cats.lower())
    title = ""
    if isinstance(row, pd.Series) and "title" in row and isinstance(row["title"], str):
        title = row["title"].lower()
    text = " ".join(tokens + [title])
    rules = [
        ("all-body", ["dress", "jumpsuit", "gown", "romper"]),
        ("outerwear", ["jacket", "coat", "trench", "parka", "cardigan", "blazer", "windbreaker", "down"]),
        ("bottoms", ["pants", "trouser", "jeans", "shorts", "skirt", "culotte", "leggings"]),
        ("tops", ["top", "t-shirt", "tee", "shirt", "blouse", "sweater", "pullover", "hoodie", "tank", "camisole"]),
        ("shoes", ["shoe", "sandal", "sneaker", "boot", "heel", "pump", "loafer", "flat", "oxford"]),
        ("bags", ["bag", "handbag", "backpack", "tote", "clutch", "purse", "satchel", "duffel"]),
        ("jewellery", ["necklace", "earring", "bracelet", "ring", "jewelry", "jewellery"]),
        ("hats", ["hat", "cap", "beanie", "beret", "fedora"]),
        ("accessories", ["belt", "scarf", "glove", "sunglass", "sunglasses", "watch", "wallet", "shawl", "tie"])
    ]
    for cat, kws in rules:
        if any(k in text for k in kws):
            return cat
    return "unknown"

def main_category(item_id: str) -> str:
    if item_id not in meta_df.index:
        return "unknown"
    row = meta_df.loc[item_id]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    sc = row.get("semantic_category", None)
    if isinstance(sc, str) and sc.strip():
        return sc.strip()
    return _infer_from_text(row)

def item_info(item_id: str) -> dict:
    if item_id not in meta_df.index:
        return {"id": item_id, "title": "(missing)", "category": "unknown"}
    row = meta_df.loc[item_id]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    for key in ["title", "description", "url_name", "name"]:
        val = row.get(key, None)
        if isinstance(val, str) and val.strip():
            title = val.strip()
            break
    else:
        title = "(no title)"
    cat = row.get("semantic_category", None)
    if not isinstance(cat, str) or not cat.strip():
        cat = _infer_from_text(row)
    return {"id": item_id, "title": title, "category": cat}

def extract_json_block(text: str) -> Optional[dict]:
    match = re.search(r'"activity"\s*:\s*"([^"]+)"', text)
    if match:
        return {"activity": match.group(1).strip()}
    json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except:
            pass
    return None


# ===== Outfit 標題（沿用你的自動蒐集器） =====
def _parse_titles_json(js) -> Dict[str, str]:
    m = {}
    if isinstance(js, dict):
        for k, v in js.items():
            if isinstance(v, str) and v.strip():
                m[str(k)] = v.strip()
            elif isinstance(v, dict):
                t = v.get("title") or v.get("name") or v.get("outfit_title") or v.get("set_title") or v.get("url_name") or v.get("description")
                if t: m[str(k)] = str(t).strip()
    elif isinstance(js, list):
        for rec in js:
            if not isinstance(rec, dict):
                continue
            gid = rec.get("set_id") or rec.get("outfit_id") or rec.get("id") or rec.get("sid") or rec.get("qid")
            title = rec.get("title") or rec.get("name") or rec.get("outfit_title") or rec.get("set_title") or rec.get("url_name") or rec.get("description")
            if gid and title:
                m[str(gid)] = str(title).strip()
    return m

def load_outfit_titles_map(base_dir: str) -> Dict[str, str]:
    candidates = [
        os.path.join(base_dir, "polyvore_outfits", "nondisjoint", "polyvore_outfit_titles.json"),
        os.path.join(base_dir, "polyvore_outfits", "disjoint",    "polyvore_outfit_titles.json"),
        os.path.join(base_dir, "polyvore_outfits", "polyvore_outfit_titles.json"),
    ]
    titles = {}
    hit_files = []
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    js = json.load(f)
                part = _parse_titles_json(js)
                if part:
                    titles.update(part)
                    hit_files.append(p)
            except Exception as e:
                print(f"[WARN] load titles {p} failed:", e)
    if not titles:
        for p in glob.glob(os.path.join(base_dir, "polyvore_outfits", "**", "*title*.json"), recursive=True):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    js = json.load(f)
                part = _parse_titles_json(js)
                if part:
                    titles.update(part)
                    hit_files.append(p)
            except:
                pass
    print(f"[INFO] outfit titles loaded: {len(titles)} entries from {len(hit_files)} file(s).")
    return titles

OUTFIT_TITLES = load_outfit_titles_map(dataset_root)

# ========== Prompt 模板 ==========
SYSTEM_PROMPT = """
You are a fashion-domain outfit and activity matching expert.

Think like a person planning their day: infer mobility, environment, and occasion from what’s worn, then pick exactly one activity from the provided list.

Core reasoning (no examples):
- Mobility intuition: Use footwear properties (sole/heel/stability/coverage/traction/contact area) to estimate safe movement and standing duration.
- Environment fit: From fabric weight/breqthability, exposure, layering/protection, and dirt/wet tolerance, infer suitability for indoor climate control, urban paved surfaces, natural terrain, or near-water settings.
- Occasion parity: The activity’s social formality must match the outfit’s level.
- Load & free-hands: Align with carry system and hand freedom.
- Strongest signal rule: When signals conflict, satisfy occasion parity first; then let the single strongest physical signal decide movement intensity and setting. Do not average toward the middle.

Decision steps:
1) Occasion Parity Gate
2) Mobility Gate
3) Environment Gate
4) Selection rule: From “Available Activities”, choose a single option that passes all three gates, is the most specific valid description, and appears verbatim in the list (exact string match).

Verification (no examples):
- Confirm occasion parity, environment compatibility, and movement intensity alignment.
- If any check fails, switch to the next most specific valid listed option that passes all checks. Prefer specificity over umbrella terms.

Strict output:
- PART A: 3–5 bullet points covering movement intensity, stability/ground, occasion parity, environment fit, and—if relevant—load/free-hands.
- PART B: {"activity": "<MUST COPY EXACTLY from Available Activities above, no paraphrasing>"}.
"""

def create_user_prompt(outfit_desc: str, items_block: str, activities_text: str, failed_activity: str = None) -> str:
    base = f"""
    Outfit description:
    {outfit_desc}

    Items:
    {items_block}

    Available Activities (choose exactly one; exact string match required):
    {activities_text}

    Instructions:
    - Apply the SYSTEM_PROMPT reasoning and decision steps without restating them.
    - Produce the strict output format:
      - PART A: 3–5 bullets (movement intensity; stability/ground; occasion parity; environment fit; load/free-hands if relevant).
      - PART B: {{"activity": "<MUST COPY EXACTLY from Available Activities above, no paraphrasing>"}}
    - Do not invent or paraphrase activity names. No examples in the explanation.
    """.strip()

    if failed_activity:
      base += f"""

      IMPORTANT (Hard Constraint):
      - The previous answer "{failed_activity}" is INVALID.
      - Do NOT output "{failed_activity}" again.
      - If you think "{failed_activity}" is conceptually right, COPY the closest exact phrase from the list that mentions the same concept verbatim.
      """.rstrip()

    return base



# ========== 載入 valid.json ========== # Changed from train.json
def _load_valid_records() -> List[dict]: # Changed function name
    candidates = [
        os.path.join(SPLIT_DIR , "valid.json"), # Changed from train.json
    ]
    for p in candidates:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                js = json.load(f)
            # valid.json 可能是 list 或 dict 裡的 list # Changed from train.json
            if isinstance(js, list):
                return js
            elif isinstance(js, dict):
                # 嘗試抓常見鍵
                for k in ("valid", "items", "outfits", "data", "records"): # Changed from train
                    if k in js and isinstance(js[k], list):
                        return js[k]
            raise ValueError(f"無法解析 valid.json 結構：{p}") # Changed from train.json
    raise FileNotFoundError("找不到 valid.json，請確認 disjoint 或 nondisjoint 目錄下是否存在。") # Changed from train.json

valid_records = _load_valid_records() # Changed variable name and function call
print(f"[INFO] valid.json 載入 {len(valid_records)} 筆") # Changed from train.json

def save_record_as_json(rec_out: dict, path: str):
    data = []
    # 如果檔案已存在，先讀取整個 JSON array
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"[WARN] 無法讀取既有檔案 {path}，將重新建立: {e}")
            data = []

    # 加入新的記錄
    data.append(rec_out)

    # 覆寫存回合法 JSON array
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# ========== 單筆處理 ==========
def _extract_outfit(rec: dict) -> Tuple[str, List[str]]:
    """回傳 (outfit_id, item_ids)；盡量相容不同欄位命名"""
    gid = str(rec.get("set_id") or rec.get("outfit_id") or rec.get("id") or rec.get("sid") or rec.get("qid") or rec.get("uid") or "")
    items = rec.get("items") or rec.get("products") or rec.get("images") or rec.get("members") or []
    item_ids = []
    if isinstance(items, list):
        for it in items:
            if isinstance(it, dict):
                iid = it.get("item_id") or it.get("id") or it.get("product_id") or it.get("member_id")
            else:
                iid = it
            if iid is not None:
                item_ids.append(str(iid))
    return gid, item_ids

def process_one_outfit(sample_idx: int, rec: dict) -> bool:
    gid, item_ids = _extract_outfit(rec)
    if not item_ids:
        print(f"[WARN] outfit {gid or sample_idx} 無 item_ids，跳過")
        return False

    items_info = [item_info(i) for i in item_ids]
    items_block = "\n".join([f"- id={it['id']} | category={it['category']} | desc={it['title']}" for it in items_info])

    outfit_title = (OUTFIT_TITLES.get(gid) or rec.get("title") or rec.get("name") or rec.get("set_title") or rec.get("description"))
    outfit_desc = outfit_title.strip() if isinstance(outfit_title, str) and outfit_title.strip() \
                  else "Outfit with: " + ", ".join(sorted(set(it["category"] for it in items_info)))

    #print("=== 單品資訊 (items_info) ===")
    #print(items_block)
    #print("\n=== 套裝描述 (outfit_desc) ===")
    #print(outfit_desc)
    # 拼貼圖
    collage_path = None
    if MAKE_COLLAGE:
        imgs = [image_path(i) for i in item_ids]
        os.makedirs("collages_valid_demo", exist_ok=True) # Changed directory name
        collage_path = make_collage([p for p in imgs if p], f"collages_valid_demo/{gid or sample_idx}.jpg", cols=4) # Changed directory name
        if collage_path:
            img = Image.open(collage_path)
            plt.figure(figsize=(6,6))
            plt.imshow(img)
            plt.axis("off")
            plt.title("VALID Outfit Collage") # Changed title
            plt.show()

    # 重試機制
    max_retry = 5
    failed_activity = None

    for attempt in range(max_retry):
        print(f"\n--- 嘗試第 {attempt+1} 次 ---")
        torch.cuda.empty_cache()

        user_prompt = create_user_prompt(
            outfit_desc=outfit_desc,
            items_block=items_block,
            activities_text=activities_text,
            failed_activity=failed_activity if FEEDBACK_ON_FAIL else None
        )

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": []}
        ]
        if collage_path:
            messages[1]["content"].append({"type": "image", "image": collage_path})
        messages[1]["content"].append({"type": "text", "text": user_prompt})

        try:
            output = pipe(text=messages, max_new_tokens=1000)
            raw_text = output[0]["generated_text"][-1]["content"]
            print(f"\n=== MODEL RAW RESPONSE (attempt {attempt+1}) ===")
            print(raw_text)
            print("="*50)
        except Exception as e:
            print(f"推論失敗 (第{attempt+1}次)：{e}")
            time.sleep(1)
            continue

        #print(f"\n=== MODEL RAW RESPONSE (attempt {attempt+1}) ===")
        #print(raw_text)
        #print("="*50)

        parsed = extract_json_block(raw_text)
        activity = None
        met_value = None

        if isinstance(parsed, dict):
            activity = parsed.get("activity")
            print(activity)
            if activity:
                # 精確比對
                if activity in activity2met:
                    met_value = activity2met[activity]
                    print(f"[SUCCESS] 找到精確匹配活動: {activity} (MET: {met_value})")
                # 模糊比對
                elif ALLOW_FUZZY:
                    match = get_close_matches(activity, list(activity2met.keys()), n=1, cutoff=FUZZY_CUTOFF)
                    if match:
                        activity = match[0]
                        met_value = activity2met[activity]
                        print(f"[SUCCESS] 找到模糊匹配活動: {activity} (MET: {met_value})")

        if activity is not None and met_value is not None:
            # 保存結果（逐行 JSONL）
            rec_out = {
                        "set_id": gid,
                        "activity": activity,
                        "met_value": met_value,
                    }
            save_record_as_json(rec_out, OUT_JSON)
            print(f"[INFO] outfit {gid or sample_idx} 處理成功，共嘗試 {attempt+1} 次")
            return True

        failed_activity = parsed.get("activity") if isinstance(parsed, dict) else "(無法解析)"
        print(f"[WARN] 第 {attempt+1} 次失敗：activity='{failed_activity}' 無效或不在清單中")

    print(f"[ERROR] outfit {gid or sample_idx} 處理失敗，已達最大重試次數")
    return False

# 已完成的 set_id
completed_set_ids = set()

if os.path.exists(OUT_JSON):
    with open(OUT_JSON, "r", encoding="utf-8") as f:
       js = json.load(f)   # 整個 JSON array
       for rec in js:
         sid = rec.get("set_id")
         if sid:
           completed_set_ids.add(str(sid))

print(f"[INFO] 已有 {len(completed_set_ids)} 筆完成，將從中斷點繼續")

def main():
    n = None
    # 全部資料的 set_id
    all_set_ids = []
    for rec in valid_records: # Changed from train_records
        gid, _ = _extract_outfit(rec)
        if gid:
            all_set_ids.append(str(gid))

    # 剩下還沒完成的
    remaining_indices = [
        i for i, rec in enumerate(valid_records) # Changed from train_records
        if str(_extract_outfit(rec)[0]) not in completed_set_ids
    ]

    already_done = len(completed_set_ids)
    target_new = len(remaining_indices) if n is None else min(n, len(remaining_indices))
    # 進度列分母：已完成 + 本輪目標
    goal_total = already_done + target_new

    print(f"[INFO] 已完成: {already_done}，本輪目標新增: {target_new}，資料總筆數: {len(valid_records)}") # Changed from train_records
    success_new = 0
    tried_new = 0

    with tqdm(total=goal_total, initial=already_done, desc="Processing VALID outfits") as pbar: # Changed description
        for i in remaining_indices:
            if n is not None and success_new >= n:
                break

            try:
                succeeded = process_one_outfit(i, valid_records[i]) # Changed from train_records
                tried_new += 1

                # 只有成功才前進進度條
                if succeeded:
                    success_new += 1
                    pbar.update(1)

            except Exception as e:
                print(f"[ERROR] outfit idx={i} 處理異常: {e}")


    print(f"\n[FINAL] 本輪成功 {success_new}/{target_new}，累計完成 {already_done + success_new}/{goal_total}")

if __name__ == "__main__":
    main()